---

# 스프린트미션16 4팀_김명환

In [1]:
!pip install --index-url https://test.pypi.org/simple/ helper-plot-hangul
!pip install --index-url https://test.pypi.org/simple/ helper-utils

Looking in indexes: https://test.pypi.org/simple/
Looking in indexes: https://test.pypi.org/simple/


In [2]:
# import importlib
# from helper_plot_hangul import helper_plot_hangul
# importlib.reload(helper_plot_hangul)

# import helper_utils.helper_logger as helper_logger
# importlib.reload(helper_logger)

# import helper_utils.helper_utils_colab as helper_utils_colab
# importlib.reload(helper_utils_colab)

from helper_plot_hangul import *
from helper_utils.helper_logger import *
from helper_utils.helper_utils_colab import *
from helper_utils.helper_utils_print import *

2025-12-06 11:43:56 D [helper_utils_colab:203] - my_driver(): D:\GoogleDrive (real path: D:\GoogleDrive)
2025-12-06 11:43:56 D [helper_utils_colab:71] - Found MY_CACHE_LOCAL from .env: d:\temp\cache_local


In [3]:
# 기본 라이브러리

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics import average_precision_score

# --- 기타 라이브러리 ---
from PIL import Image
from PIL import ImageFilter
from PIL import ImageDraw
import albumentations as A
import IPython.display
#from tqdm import tqdm
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# --- PyTorch: 딥러닝 관련 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as TF
from torch.nn import CrossEntropyLoss
from collections import OrderedDict

# --- 기타 ---
import re
import os
import sys
import copy
import json
import math
import random
import yaml
import shutil
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from datetime import datetime
from datetime import timezone, timedelta
import pytz
__kst = pytz.timezone('Asia/Seoul')

# GPU 설정
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

  # 재현 가능한 결과를 위해
np.random.seed(42)
torch.manual_seed(42)
if __device == 'cuda':
    torch.cuda.manual_seed_all(42)

print(f"라이브러리 로드 완료 사용장치:{__device}")

라이브러리 로드 완료 사용장치:cpu


### > 설정 < 플레그

In [4]:
DEBUG_ON = False if IS_COLAB else True
DEBUG_ON = False
TRAIN_ON = False
logger.info(f"IS_COLAB={IS_COLAB}")
logger.info(f"DEBUG_ON={DEBUG_ON}")


2025-12-06 11:44:01 I [helper_utils_print:4] - IS_COLAB=False
2025-12-06 11:44:01 I [helper_utils_print:5] - DEBUG_ON=False


In [5]:
def get_tqdm_kwargs():
    """Widget 오류를 방지하는 안전한 tqdm 설정"""
    return {
        'disable': False,
        'leave': True,
        'file': sys.stdout,
        'ascii': True,  # ASCII 문자만 사용
        'dynamic_ncols': False,
#        'ncols': 80  # 고정 폭
    }

def get_path_modeling(add_path = None):
    modeling_path = "modeling16"
    if DEBUG_ON:
        modeling_path = modeling_path +"_debug"
    return my_driver_path(modeling_path)

def get_path_modeling_release(add_path = None):
    modeling_path = "modeling16"
    if DEBUG_ON:
        modeling_path = modeling_path
    return my_driver_path(modeling_path)

def save_model_dict(model, path, pth_name, kwargs=None):
    """모델 state_dict와 추가 정보를 저장"""
    def safe_makedirs(path):
        """안전한 디렉토리 생성"""
        if os.path.exists(path) and not os.path.isdir(path):
            os.remove(path)  # 파일이면 삭제
        os.makedirs(path, exist_ok=True)

    # 디렉토리 생성
    safe_makedirs(path)

    # 모델 구조 정보 추출
    model_info = {
        'class_name': model.__class__.__name__,
        'init_args': {},
        'str': str(model),
        'repr': repr(model),
        'modules': [m.__class__.__name__ for m in model.modules()],
    }

    # 생성자 인자 자동 추출(가능한 경우)
    if hasattr(model, '__dict__'):
        for key in ['in_ch', 'base_ch', 'num_classes', 'out_ch']:
            if hasattr(model, key):
                model_info['init_args'][key] = getattr(model, key)

    # kwargs 처리
    extra_info = {}
    if kwargs is not None:
        if isinstance(kwargs, str):
            extra_info = json.loads(kwargs)
        elif isinstance(kwargs, dict):
            extra_info = kwargs

    model_info.update(extra_info)

    # 저장할 dict 구성
    save_dict = {
        'model_state': model.state_dict(),
        'class_name': model.__class__.__name__,
        'model_info': model_info,
    }

    save_path = os.path.join(path, f"{pth_name}.pth")
    torch.save(save_dict, save_path)
    return save_path

def load_model_dict(path, pth_name=None):
    """
    save_model_dict로 저장한 모델을 불러오는 함수
    반환값: (model_state, model_info)
    """
    import torch
    load_path = path
    if pth_name is not None:
        load_path = os.path.join(path, f"{pth_name}.pth")
    checkpoint = torch.load(load_path, map_location='cpu', weights_only=False)  # <-- 여기 추가
    model_state = checkpoint.get('model_state')
    model_info = checkpoint.get('model_info')
    model_info['file_name'] = os.path.basename(load_path)
    return model_state, model_info


def search_pth_files(base_path):
    """
    입력된 경로의 하위 폴더들에서 pth 파일들을 검색
    """
    pth_files = []

    if not os.path.exists(base_path):
        print(f"경로가 존재하지 않습니다: {base_path}")
        return pth_files

    print(f"pth 파일 검색 시작: {base_path}")

    # 하위 폴더들을 순회하며 pth 파일 검색
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith('.pth'):
                pth_path = os.path.join(root, file)
                pth_files.append(pth_path)

    # 결과 정리 및 출력
    if pth_files:
        print(f"\n발견된 pth 파일들 ({len(pth_files)}개):")
        for i, pth_file in enumerate(pth_files, 1):
            # 상대 경로로 표시 (base_path 기준)
            rel_path = os.path.relpath(pth_file, base_path)
            print(f" {i:2d}. {rel_path}")
    else:
        print("pth 파일을 찾을 수 없습니다.")

    return pth_files

print("유틸리티 함수 로드 완료")

유틸리티 함수 로드 완료


#### 1.2.1 yolo 유틸리티 함수

In [6]:

# yolo_dataset_path

# C:\Users\sw1\.cache\kagglehub\datasets\devdgohil\the-oxfordiiit-pet-dataset\versions\2\yolo_dataset
def create_yolo_dataset_yaml(train_df, valid_df, test_df, yolo_dataset_path, ignore=False):
    """YOLO 형식의 데이터셋 yaml 파일 생성"""

    # YOLO 데이터셋 폴더 생성
    if os.path.exists(yolo_dataset_path):
        if ignore:
            print(f"Yolo 데이터셋 삭제 {yolo_dataset_path}")
            shutil.rmtree(yolo_dataset_path)
        else:
            print(f"Yolo 데이터셋 있음 {yolo_dataset_path}")
            print_dir_tree(yolo_dataset_path)
            yaml_path = os.path.join(yolo_dataset_path, "dataset.yaml")
            return yaml_path

    os.makedirs(yolo_dataset_path, exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_path, "inf"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_path, "trimaps"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_path, "images", "train"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_path, "images", "val"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_path, "images", "test"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_path, "labels", "train"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_path, "labels", "val"), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_path, "labels", "test"), exist_ok=True)

    # 이미지 복사 및 라벨 파일 생성
    def convert_to_yolo_format(df, split_name):
        pbar = tqdm(df.iterrows(), total=len(df), desc=f"yolo dataset {split_name}", **get_tqdm_kwargs())
        for _, row in pbar:
            # 이미지 복사
            src_img = row['image_path']
            if os.path.exists(src_img):
                dst_img = os.path.join(yolo_dataset_path, "images", split_name, f"{row['image_name']}.jpg")
                shutil.copy2(src_img, dst_img)

                src_trimaps = row['trimap_path']
                if os.path.exists(src_trimaps):
                    dst_trimaps = os.path.join(yolo_dataset_path, "trimaps", f"{row['image_name']}.png")
                    shutil.copy2(src_trimaps, dst_trimaps)

                # bbox 정보가 있는지 확인
                has_bbox = not pd.isna(row.get('xmin'))

                # Label 파일 생성 (bbox 정보가 있을 때만)
                if has_bbox:
                    img_width = row['width']
                    img_height = row['height']

                    # bbox 좌표를 YOLO 형식으로 변환
                    x_center = (row['xmin'] + row['xmax']) / 2 / img_width
                    y_center = (row['ymin'] + row['ymax']) / 2 / img_height
                    bbox_width = (row['xmax'] - row['xmin']) / img_width
                    bbox_height = (row['ymax'] - row['ymin']) / img_height

                    # 클래스 ID (species_id - 1, 0: cat, 1: dog)
                    class_id = row['species_id'] - 1

                    # 라벨 파일 생성
                    label_file = os.path.join(yolo_dataset_path, "labels", split_name, f"{row['image_name']}.txt")
                    with open(label_file, 'w') as f:
                        f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}\n")
                else:
                    # bbox 정보가 없으면 빈 라벨 파일 생성
                    label_file = os.path.join(yolo_dataset_path, "labels", split_name, f"{row['image_name']}.txt")
                    open(label_file, 'w').close()

                # inf 파일 생성 (모든 split에 대해)
                inf_file = os.path.join(yolo_dataset_path, "inf", f"{row['image_name']}.inf")
                with open(inf_file, 'w', encoding='utf-8') as f:
                    f.write(f"image_name: {row['image_name']}\n")
                    f.write(f"kind_id: {row['class_id']}\n")
                    f.write(f"species_id: {row['species_id']}\n")
                    f.write(f"breed_id: {row['breed_id']}\n")
                    f.write(f"species: {row['species']}\n")


    # 데이터 변환
    convert_to_yolo_format(train_df, "train")
    convert_to_yolo_format(valid_df, "val")
    convert_to_yolo_format(test_df, "test")

    # dataset.yaml 파일 생성
    dataset_config = {
        'path': yolo_dataset_path,
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': 2,  # 클래스 수
        'names': ['cat', 'dog']
    }

    yaml_path = os.path.join(yolo_dataset_path, "dataset.yaml")
    with open(yaml_path, 'w') as f:
        yaml.dump(dataset_config, f, default_flow_style=False)

    print(f"YOLO 데이터셋 생성 완료: {yolo_dataset_path}")
    return yaml_path

def yolo_dataset_to_dataframe(yaml_path):
    """
    YOLO dataset.yaml 파일을 참고하여 train/val/test DataFrame 생성 + inf 폴더 정보 추가
    """


    # yaml 파일 로드
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    base_path = config['path']
    splits = ['train', 'val', 'test'] if 'test' in config else ['train', 'val']
    dfs = {}

    for split in splits:
        img_dir = os.path.join(base_path, config[split])
        label_dir = os.path.join(base_path, 'labels', split) if split != 'test' else None
        inf_dir = os.path.join(base_path, 'inf') if os.path.exists(os.path.join(base_path, 'inf')) else None
        trimaps_dir = os.path.join(base_path, 'trimaps') if os.path.exists(os.path.join(base_path, 'trimaps')) else None

        img_files = [f for f in os.listdir(img_dir) if f.lower().endswith('.jpg')]
        records = []
        pbar = tqdm(img_files, total=len(img_files), desc=f"DataFrame from yolo dataset {split}", **get_tqdm_kwargs())
        for img_file in pbar:
            image_path = os.path.join(img_dir, img_file)
            image_name = os.path.splitext(img_file)[0]
            label_path = None
            class_id = None
            bbox = None
            inf_data = {}
            trimap_file = None

            if label_dir:
                label_file = os.path.join(label_dir, f"{image_name}.txt")
                if os.path.exists(label_file):
                    with open(label_file, 'r') as lf:
                        lines = lf.readlines()
                        if lines:
                            parts = lines[0].strip().split()
                            if len(parts) == 5:
                                class_id = int(parts[0])
                                x_center = float(parts[1])
                                y_center = float(parts[2])
                                w = float(parts[3])
                                h = float(parts[4])
                                bbox = (x_center, y_center, w, h)
                                label_path = label_file

            # inf 폴더 정보 추가
            if inf_dir:
                inf_file = os.path.join(inf_dir, f"{image_name}.inf")
                if os.path.exists(inf_file):
                    with open(inf_file, 'r', encoding='utf-8') as f_inf:
                        for line in f_inf:
                            if ':' in line:
                                k, v = line.strip().split(':', 1)
                                inf_data[k.strip()] = v.strip()

            if trimaps_dir:
                trimap_file = os.path.join(trimaps_dir, f"{image_name}.png")
                if os.path.exists(trimap_file) is False:
                    trimap_file = None

            record = {
                'image_name': image_name,
            }
            record.update({
                'split': split,
                'class_id': class_id,
                'bbox': bbox,
            })
            if inf_data:
                record.update(inf_data)
            record.update({
                'image_path': image_path,
                'label_path': label_path,
                'trimap_path': trimap_file,
            })
            records.append(record)

        dfs[split] = pd.DataFrame(records)

    train_df = dfs.get('train', None)
    valid_df = dfs.get('val', None)
    test_df = dfs.get('test', None)

    print(f"train_df: {train_df.shape}, valid_df: {valid_df.shape}, test_df: {test_df.shape}")
    return train_df, valid_df, test_df

def yolo_to_coco_bbox(yolo_bbox, img_width=224, img_height=224):
    """
    yolo_bbox: [x_center, y_center, w, h] (정규화된 값)
    반환: [xmin, ymin, xmax, ymax] (pixel 좌표)
    """
    x_center, y_center, w, h = yolo_bbox
    xmin = (x_center - w / 2) * img_width
    ymin = (y_center - h / 2) * img_height
    xmax = (x_center + w / 2) * img_width
    ymax = (y_center + h / 2) * img_height
    return [xmin, ymin, xmax, ymax]


def update_yaml_paths_to_absolute(yaml_path):
    """YAML 파일의 상대 경로를 절대 경로로 업데이트합니다.

    Args:
        yaml_path (str): 업데이트할 YAML 파일 경로
    """
    # dataset.yaml 파일을 읽어서 내부 경로(path)를 절대경로로 변환
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    yaml_dir = os.path.dirname(yaml_path)
    # 'path' 항목을 yaml 파일 위치 기준 절대경로로 변경
    data['path'] = os.path.normpath(os.path.join(yaml_dir, data['path']))
    # 아래 코드는 필요시 train/val/test 경로도 절대경로로 변경 가능
    # for key in ['train', 'val', 'test']:
    #     if key in data and not os.path.isabs(data[key]):
    #         data[key] = os.path.normpath(os.path.join(yaml_dir, data[key]))

    # 변경된 내용을 다시 yaml 파일에 저장
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, allow_unicode=True)


print("yolo 유틸리티 함수 로드 완료")

yolo 유틸리티 함수 로드 완료


## 데이터 로드

In [7]:
import kagglehub
kagglehub_path = kagglehub.dataset_download("devdgohil/the-oxfordiiit-pet-dataset")
def get_path_data():
    path = kagglehub_path
    return path

print("Path to dataset files:", get_path_data())
print_dir_tree(kagglehub_path)


Path to dataset files: C:\Users\sw1\.cache\kagglehub\datasets\devdgohil\the-oxfordiiit-pet-dataset\versions\2
2025-12-06 11:44:02 I [helper_utils_print:127] - |-- annotations
2025-12-06 11:44:02 I [helper_utils_print:134] -    [데이터파일: 1개]
2025-12-06 11:44:02 I [helper_utils_print:127] -    |-- annotations
2025-12-06 11:44:02 I [helper_utils_print:134] -       [데이터파일: 7개]
2025-12-06 11:44:02 I [helper_utils_print:148] -       |-- ._trimaps
2025-12-06 11:44:02 I [helper_utils_print:148] -       |-- README
2025-12-06 11:44:02 I [helper_utils_print:148] -       |-- list.txt
2025-12-06 11:44:02 I [helper_utils_print:127] -       |-- trimaps
2025-12-06 11:44:03 I [helper_utils_print:134] -          [데이터파일: 14780개]
2025-12-06 11:44:04 I [helper_utils_print:148] -          |-- ._Abyssinian_1.png
2025-12-06 11:44:04 I [helper_utils_print:148] -          |-- ._Abyssinian_10.png
2025-12-06 11:44:04 I [helper_utils_print:148] -          |-- ._Abyssinian_100.png
2025-12-06 11:44:05 I [helper_utils_

### 2.2. xml 데이터 DataFrame

In [8]:

def get_pet_dataset_paths():
    """Oxford-IIIT Pet Dataset 경로 반환 (실제 xmls 경로 자동 탐색)"""
    data_path = get_path_data()
    images_path = os.path.join(data_path, "images", "images")
    annotations_path = os.path.join(data_path, "annotations", "annotations")

    # xmls 경로 후보 리스트
    xmls_candidates = [
        os.path.join(annotations_path, "xmls"),
        os.path.join(annotations_path, "annotations", "xmls"),
        os.path.join(data_path, "annotations", "xmls"),
        os.path.join(data_path, "annotations", "annotations", "xmls"),
        annotations_path,
    ]
    xmls_path = None
    for candidate in xmls_candidates:
        if os.path.exists(candidate):
            xmls_path = candidate
            break

    dataset_path = {
        "root": data_path,
        "images": images_path,
        "annotations": annotations_path,
        "xmls": xmls_path if xmls_path else "",  # 없으면 빈 문자열
    }
    return dataset_path

__dataset_path = get_pet_dataset_paths()

def data_path(key):
    return __dataset_path.get(key)

#print("Path to dataset files:", get_path_data())
#print_dir_tree(kagglehub_path, max_depth=3)
print(json.dumps(__dataset_path, indent=2, ensure_ascii=False))
# for key in __dataset_path.keys():
#     print(f"{key} 데이터 경로: {data_path(key)}")


{
  "root": "C:\\Users\\sw1\\.cache\\kagglehub\\datasets\\devdgohil\\the-oxfordiiit-pet-dataset\\versions\\2",
  "images": "C:\\Users\\sw1\\.cache\\kagglehub\\datasets\\devdgohil\\the-oxfordiiit-pet-dataset\\versions\\2\\images\\images",
  "annotations": "C:\\Users\\sw1\\.cache\\kagglehub\\datasets\\devdgohil\\the-oxfordiiit-pet-dataset\\versions\\2\\annotations\\annotations",
  "xmls": "C:\\Users\\sw1\\.cache\\kagglehub\\datasets\\devdgohil\\the-oxfordiiit-pet-dataset\\versions\\2\\annotations\\annotations\\xmls"
}


- list.txt 파싱

In [9]:
def parse_list_file(file_path):
    """
    list 파일을 파싱하여 DataFrame으로 변환
    list.txt
    #Image CLASS-ID SPECIES BREED ID
    #ID: 1:37 Class ids
    #SPECIES: 1:Cat 2:Dog
    #BREED ID: 1-25:Cat 1:12:Dog
    #All images with 1st letter as captial are cat images
    #images with small first letter are dog images
    Abyssinian_100 1 1 1
    Abyssinian_101 1 1 1
    Abyssinian_102 1 1 1
    Abyssinian_103 1 1 1

    """
    data = []

    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split()
            if len(parts) == 4:
                image_name = parts[0]
                class_id = int(parts[1])
                species_id = int(parts[2])  # 1: cat, 2: dog
                breed_id = int(parts[3])

                data.append({
                    'image_name': image_name,
                    'class_id': class_id,
                    'species_id': species_id,
                    'breed_id': breed_id,
                    'species': 'cat' if species_id == 1 else 'dog'
                })

    return pd.DataFrame(data)

def load_xmls(list_df, train_df, test_df):
    """간단하게 XML 로드하고 train/test 분할"""
    xml_dir = __dataset_path['xmls']
    records = []

    endmesg = ""
    # XML 파일들 읽기
    pbar = tqdm(list_df.iterrows(), total=len(list_df), **get_tqdm_kwargs())
    for _, row in pbar:
        image_name = row['image_name']
        xml_file = os.path.join(xml_dir, image_name + '.xml')

        pbar.set_postfix_str(endmesg)

        # 이미지와 XML 파일 경로 설정
        image_path = os.path.join(__dataset_path['images'], f"{image_name}.jpg")
        if os.path.exists(image_path) == False:
            endmesg = f"파일 없음: {image_path}"
            continue

        #  trimap 경로 수정 - 여러 패턴으로 찾기
        trimaps_dir = os.path.join(__dataset_path['annotations'], 'trimaps')
        trimaps = ""
        if os.path.exists(trimaps_dir):
            # 여러 가능한 파일명 패턴 시도
            possible_names = [
                f"{image_name}.png",           # 기본 패턴
                f"trimap_{image_name}.png",    # trimap_ 접두사
                f"{image_name}_trimap.png"    # _trimap 접미사
            ]

            for pattern in possible_names:
                trimap_path = os.path.join(trimaps_dir, pattern)
                if os.path.exists(trimap_path):
                    # 숨김 파일이 아니고 크기가 충분한 파일만 선택
                    if not pattern.startswith('._') and os.path.getsize(trimap_path) > 1000:
                        try:
                            # 실제로 이미지인지 확인
                            test_img = Image.open(trimap_path)
                            test_img.verify()
                            trimaps = trimap_path
                            break
                        except:
                            continue

        field = {
                'image_name': image_name,
                'image_path': image_path,
                'class_id': row['class_id'],
                'species_id': row['species_id'],
                'breed_id': row['breed_id'],
                'species': row['species'], # cat dog
                'trimap_path': trimaps,
                'width': None,
                'height': None,
                'xmin': None,
                'ymin': None,
                'xmax': None,
                'ymax': None,
                'object_name': None,
                'pose': None,
                'truncated': None,
                'occluded': None,
                'difficult': None,
                'augmented': 0, # 0 오리지널 1 ~ n 증강옵션에 따라
                }

        if trimaps is None or trimaps == "":
            endmesg = f"Trimap 없음: {image_name} trimaps:{trimaps}"
            continue

        if os.path.exists(xml_file):
            try:
                # XML 파싱
                tree = ET.parse(xml_file)
                root = tree.getroot()
                size = root.find('size')
                width = int(size.find('width').text)
                height = int(size.find('height').text)

                # 바운딩 박스 정보
                for obj in root.findall('object'):
                    bbox = obj.find('bndbox')
                    field.update({
                        'width': width,
                        'height': height,
                        'xmin': int(bbox.find('xmin').text),
                        'ymin': int(bbox.find('ymin').text),
                        'xmax': int(bbox.find('xmax').text),
                        'ymax': int(bbox.find('ymax').text),
                        'object_name': obj.find('name').text if obj.find('name') is not None else None, # cat dog
                        'pose': obj.find('pose').text if obj.find('pose') is not None else None,
                        'truncated': int(obj.find('truncated').text) if obj.find('truncated') is not None else None,
                        'occluded': int(obj.find('occluded').text) if obj.find('occluded') is not None else None,
                        'difficult': int(obj.find('difficult').text) if obj.find('difficult') is not None else None,
                    })

                    # bbox 정보가 맞는지 검토 하자
                    if (field['xmin'] < 0 or field['ymin'] < 0 or
                        field['xmax'] > field['width'] or field['ymax'] > field['height']):
                        endmesg = f"Invalid bbox for {image_name}: {field}"
                        continue
                    if train_df['image_name'].isin([image_name]).any(): # Train 데이터에 존재
                        records.append(field.copy())
                    else:
                        endmesg = f"Train 데이터에 없음: {image_name}"

            except Exception as e:
                endmesg = print(f"Error parsing {xml_file}: {e}")
        else:
            if test_df['image_name'].isin([image_name]).any():
                records.append(field.copy())
            else:
                endmesg = f"Test 데이터에 없음: {image_name}"

    # DataFrame 생성
    df = pd.DataFrame(records)

    # 간단한 train/test 분할
    df['train'] = df['image_name'].isin(train_df['image_name'])

    print(f"총 데이터: {len(df)}/{len(list_df)}개 ({len(df)/len(list_df)*100:.1f}%)")
    valid_trimaps = (df['trimap_path'] != "").sum()
    print(f"유효한 Trimap: {valid_trimaps}/{len(df)}개 ({valid_trimaps/len(df)*100:.1f}%)")

    return df



In [10]:
__kind_id_species = {}
def get_species_by_kind_id(class_id):
    return __kind_id_species.get(class_id, None)

def print_kind_id():
    # class ID를 DataFrame로 하여 가로로 출력하자
    df = pd.DataFrame(__kind_id_species.items(), columns=['Kind ID', 'Species'])
    df.T.head()

def load_data_frame():
    global __kind_id_species

    # 파일 경로 설정
    base_path = __dataset_path['annotations']

    list_path = os.path.join(base_path, "list.txt")
    trainval_path = os.path.join(base_path, "trainval.txt")
    test_path = os.path.join(base_path, "test.txt")

    # 데이터 파싱
    list_df = parse_list_file(list_path)
    train_df = parse_list_file(trainval_path)
    test_df = parse_list_file(test_path)

    for _, row in list_df.iterrows():
        __kind_id_species[row['class_id']] = row['species']

    data_df = load_xmls(list_df, train_df, test_df)

    # 다시 데이터 분환
    train_df = data_df[data_df['train']].copy()
    test_df = data_df[~data_df['train']].copy()
    train_df, valid_df = train_test_split(train_df, test_size=0.3, random_state=42)

    return train_df, valid_df, test_df



In [11]:
root_cache_path = my_cache()
root_my_driver = my_driver()

logger.debug(f"root_cache_path: {root_cache_path}")
logger.debug(f"root_my_driver: {root_my_driver}")

2025-12-06 11:44:07 D [helper_utils_print:4] - root_cache_path: d:\temp\cache_local
2025-12-06 11:44:07 D [helper_utils_print:5] - root_my_driver: D:\GoogleDrive


In [12]:

org_train_df, org_valid_df, org_test_df = load_data_frame()
train_df = org_train_df.copy()
valid_df_all = org_valid_df.copy()
#test_df = org_test_df.copy()

valid_df = valid_df_all[:-100]
test_df = valid_df_all[-100:]

  0%|          | 0/7349 [00:00<?, ?it/s]

총 데이터: 7323/7349개 (99.6%)
유효한 Trimap: 7323/7323개 (100.0%)


In [13]:
logger.info(f"DEBUG_ON={DEBUG_ON}")

2025-12-06 11:44:23 I [helper_utils_print:1] - DEBUG_ON=False


In [14]:
#test_df = test_df[:10]

# train_df = org_train_df[:10] if DEBUG_ON else org_train_df
# valid_df = org_valid_df[:10] if DEBUG_ON else org_valid_df[:-100]
# test_df = org_valid_df[10:20] if DEBUG_ON else org_valid_df[-100:]


In [15]:
print(f"Train 데이터: {train_df.shape[0]}개")
print(f"Valid 데이터: {valid_df.shape[0]}개")
print(f"Test 데이터: {test_df.shape[0]}개")

Train 데이터: 2564개
Valid 데이터: 1000개
Test 데이터: 100개


## Yolo DataSet

In [16]:

yolo_dataset_path = my_cache_path("yolo", "the-oxfordiiit-pet-dataset")
yaml_path = create_yolo_dataset_yaml(train_df, valid_df, test_df, yolo_dataset_path, ignore=True)
logger.debug(f"yaml_path: {yaml_path}")
update_yaml_paths_to_absolute(yaml_path)
train_df, valid_df, test_df = yolo_dataset_to_dataframe(yaml_path)

2025-12-06 11:44:23 D [helper_utils_colab:515] - my_cache_path base: D:\temp\cache_local
2025-12-06 11:44:23 D [helper_utils_colab:582] - my_cache_path result (before create/validate): D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
2025-12-06 11:44:23 D [helper_utils_colab:588] - Directory created: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
2025-12-06 11:44:23 D [helper_utils_colab:598] - Path validation passed: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
Yolo 데이터셋 삭제 D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset


yolo dataset train:   0%|          | 0/2564 [00:00<?, ?it/s]

yolo dataset val:   0%|          | 0/1000 [00:00<?, ?it/s]

yolo dataset test:   0%|          | 0/100 [00:00<?, ?it/s]

YOLO 데이터셋 생성 완료: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
2025-12-06 11:44:46 D [helper_utils_print:3] - yaml_path: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\dataset.yaml


DataFrame from yolo dataset train:   0%|          | 0/2564 [00:00<?, ?it/s]

DataFrame from yolo dataset val:   0%|          | 0/1000 [00:00<?, ?it/s]

DataFrame from yolo dataset test:   0%|          | 0/100 [00:00<?, ?it/s]

train_df: (2564, 11), valid_df: (1000, 11), test_df: (100, 11)


In [17]:
print(f"Train 데이터: {train_df.shape[0]}개")
print(f"Valid 데이터: {valid_df.shape[0]}개")
print(f"Test 데이터: {test_df.shape[0]}개")

print('yaml_path=', yaml_path)
print_dir_tree(yolo_dataset_path)


Train 데이터: 2564개
Valid 데이터: 1000개
Test 데이터: 100개
yaml_path= D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\dataset.yaml
2025-12-06 11:46:51 I [helper_utils_print:148] - |-- dataset.yaml
2025-12-06 11:46:51 I [helper_utils_print:127] - |-- images
2025-12-06 11:46:51 I [helper_utils_print:134] -    [데이터파일: 3개]
2025-12-06 11:46:51 I [helper_utils_print:127] -    |-- test
2025-12-06 11:46:51 I [helper_utils_print:134] -       [데이터파일: 100개]
2025-12-06 11:46:51 I [helper_utils_print:148] -       |-- Abyssinian_112.jpg
2025-12-06 11:46:51 I [helper_utils_print:148] -       |-- Abyssinian_121.jpg
2025-12-06 11:46:51 I [helper_utils_print:148] -       |-- Abyssinian_148.jpg
2025-12-06 11:46:52 I [helper_utils_print:154] -          ... files
2025-12-06 11:46:52 I [helper_utils_print:127] -    |-- train
2025-12-06 11:46:52 I [helper_utils_print:134] -       [데이터파일: 2564개]
2025-12-06 11:46:52 I [helper_utils_print:148] -       |-- Abyssinian_1.jpg
2025-12-06 11:46:52 I [helper_utils_print:148

In [18]:
from ultralytics import YOLO
# GPU 확인
device = __device
print(f"사용 디바이스: {device}")
print(f"CUDA 버전: {torch.version.cuda}")

사용 디바이스: cpu
CUDA 버전: None


In [19]:
os.environ['YOLO_VERBOSE'] = 'False'
os.environ['ULTRALYTICS_LOG_LEVEL'] = 'WARNING'  # 또는 'ERROR'

In [20]:
yolov8m_best_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005', 'weights', 'best.pt', create=False)
logger.info(f"yolov8m_best_path: {yolov8m_best_path}")

2025-12-06 11:46:56 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 11:46:56 D [helper_utils_colab:378] - my_driver_path result (before create/validate): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt
2025-12-06 11:46:56 D [helper_utils_colab:394] - Path validation passed: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt
2025-12-06 11:46:56 D [helper_utils_colab:396] - my_driver_path final: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt
2025-12-06 11:46:56 I [helper_utils_print:2] - yolov8m_best_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt


In [23]:
# 1. 학습된 YOLOv8 모델 로드 (YOLO CLI로 학습 후 생성된 파일)
# 예: runs/detect/pet_yolov8n/weights/best.pt
model = YOLO(yolov8m_best_path) 
output_dir_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005', 'out', create=True)
output_model_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005', 'out' , 'mission_16_yolo.pth', create=False, validate=False)
logger.info(f"output_model_path: {output_model_path}")

# 2. 모델 가중치만 추출하여 .pth 파일로 저장
# state_dict를 저장하면 .pth 포맷의 표준 방식입니다.
torch.save(model.model.state_dict(), output_model_path) 
print(f"기본 .pth 모델 저장 완료: {output_model_path}")

2025-12-06 11:53:48 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 11:53:48 D [helper_utils_colab:378] - my_driver_path result (before create/validate): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out
2025-12-06 11:53:48 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out
2025-12-06 11:53:48 D [helper_utils_colab:394] - Path validation passed: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out
2025-12-06 11:53:48 D [helper_utils_colab:396] - my_driver_path final: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out
2025-12-06 11:53:48 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 11:53:48 D [helper_utils_colab:378] - my_driver_path result (before create/validate): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out\mission_16_yolo.pth
2025-12-06 11:53:48 D [helper_utils_colab:396] - my_driver_path final

## 양자화

In [24]:
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_

In [ ]:
# 1. 학습된 모델 로드
model = YOLO(yolov8m_best_path).model.eval() # 추론 모드(eval)로 설정

# 2. 양자화 적용 (주로 Linear 레이어에 적용)
# dtype을 torch.qint8로 지정하여 8비트 정수형 양자화를 수행합니다.
quantized_model = torch.quantization.quantize_dynamic(
    model, 
    {torch.nn.Linear, torch.nn.Conv2d}, # 양자화할 모듈 타입 지정
    dtype=torch.qint8
)

output_dir_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005', 'out_qint8', create=True)
output_model_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005', 'out_qint8' , 'mission_16_yolo_quantized.pth', create=False, validate=False)
logger.info(f"output_model_path: {output_model_path}")

# 3. 양자화된 모델 저장
torch.save(quantized_model.state_dict(), output_model_path)
print(f"양자화 .pth 모델 저장 완료: {output_model_path}")

2025-12-06 12:06:48 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 12:06:48 D [helper_utils_colab:378] - my_driver_path result (before create/validate): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8
2025-12-06 12:06:48 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8
2025-12-06 12:06:48 D [helper_utils_colab:394] - Path validation passed: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8
2025-12-06 12:06:48 D [helper_utils_colab:396] - my_driver_path final: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8
2025-12-06 12:06:48 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 12:06:48 D [helper_utils_colab:378] - my_driver_path result (before create/validate): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8\mission_16_yolo_quantized.pth
2025-12-06 12:06:48 D [helper

C:\Users\sw1\AppData\Local\Temp\ipykernel_4676\2158111137.py:6: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


양자화 .pth 모델 저장 완료: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8\mission_16_yolo_quantized.pth


In [27]:
quantized_model

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
   